## Test Run

In [1]:
import torch
import torch.nn.functional as F
from text import EmbedderConfig, MedicalTextEmbedder

# 1. Your medical documents
documents = [
    "Patient presents with acute bilateral pulmonary infiltrates and hypoxia.",
    "Normal sinus rhythm without ST-segment elevation or T-wave inversion.",
    "Chest X-ray demonstrates right lower lobe consolidation consistent with pneumonia.",
    "Unremarkable brain MRI with no evidence of acute ischemia or hemorrhage.",
]

# 2. Your search query
query = "Signs of lung infection on chest imaging"


config = EmbedderConfig(
    model_dir=  "madrisight/madrimed1.2-text-embed"#"./saved_qwen3vl_text_embedder_L18",
    layers=(8,18, 8),
    batch_size=8,
    standardize=True,
    remove_pcs=0,
)

embedder = MedicalTextEmbedder(config)

embedder.index_corpus(documents)

results = embedder.search(query, top_k=4)

print(f"\nQuery: {query}\n")

for rank, (score, document) in enumerate(results, start=1):
    print(f"Rank: {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Document: {document}\n")
    

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████| 211/211 [00:01<00:00, 195.44it/s]



Query: Signs of lung infection on chest imaging

Rank: 1
Similarity: 1.4035
Document: Chest X-ray demonstrates right lower lobe consolidation consistent with pneumonia.

Rank: 2
Similarity: -0.1898
Document: Patient presents with acute bilateral pulmonary infiltrates and hypoxia.

Rank: 3
Similarity: -0.3406
Document: Unremarkable brain MRI with no evidence of acute ischemia or hemorrhage.

Rank: 4
Similarity: -0.8731
Document: Normal sinus rhythm without ST-segment elevation or T-wave inversion.



## Evaluation

In [1]:
!pip install datasets mteb pytrec_eval tqdm torch transformers scipy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 1.4 MB/s  0:00:04m 1.4 MB/s eta 0:00:01
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp311-cp311-macosx_26_0_arm64.whl size=68723 sha256=167ba7c6a02731be416c90bbeb8bc79466f42cbe5b2c96cdb5a04c9733597115
  Stored in directory: /Users/init-mac/Library/Caches/pip/wheels/0f/89/42/86aecdb99975f1840c27bc37fdfed72116abcf82e2c9dc76a8
Successfully built pytrec_eval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mteb]━━━━━━━━━━━━━ 2/3 [mteb]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [4]:


import dataclasses
import gc
import json
import logging
import math
import os
import sys
import time
import unicodedata
from collections import defaultdict
from typing import Any, Sequence
from dataclasses import dataclass, replace
from typing import Collection, Literal, Sequence
import numpy as np
import torch
from datasets import load_dataset
import torch
from text import EmbedderConfig, MedicalTextEmbedder

os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger("UNClinicalBenchmark")

# Published reference numbers (hard-coded, from the benchmark table; NOT produced by this script)
BASELINE_BGE_SMALL = {
    "model_name": "BAAI/bge-small-en-v1.5 (published)",
    "params": "33M",
    "TRECCOVID": {"nDCG@10": 66.45, "MRR@5": 87.23},
    "MedicalQARetrieval": {"nDCG@10": 65.55, "MRR@5": 59.19},
    "PublicHealthQA": {"nDCG@10": 76.17, "MRR@5": 70.60},
    "NFCorpus": {"nDCG@10": 33.70, "MRR@5": 52.19},
    "ArguAna": {"nDCG@10": 59.49, "MRR@5": 49.62},
}

BGE_QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

TASKS_ORDER = ["TRECCOVID", "MedicalQARetrieval", "PublicHealthQA", "NFCorpus", "ArguAna"]

BENCHMARK_REGISTRY = {
    "TRECCOVID": {
        "hf_repo": "mteb/trec-covid",
        "corpus_config": "corpus",
        "queries_config": "queries",
        "qrels_config": "default",
    },
    "MedicalQARetrieval": {
        "hf_repo": "mteb/medical_qa",
        "corpus_config": "corpus",
        "queries_config": "queries",
        "qrels_config": "default",
    },
    "PublicHealthQA": {
        "hf_repo": "mteb/PublicHealthQA",
        "corpus_config": "english-corpus",
        "queries_config": "english-queries",
        "qrels_config": "english-qrels",
    },
    "NFCorpus": {
        "hf_repo": "mteb/nfcorpus",
        "corpus_config": "corpus",
        "queries_config": "queries",
        "qrels_config": "default",
    },
    "ArguAna": {
        "hf_repo": "mteb/arguana",
        "corpus_config": "corpus",
        "queries_config": "queries",
        "qrels_config": "default",
    },
}

EMPTY_TEXT_PLACEHOLDER = "N/A"
EVAL_TOP_K = 10

# ---------------------------------------------------------------------------------------------------------
# LOCKED CONFIG. Edit only by deliberately changing this block; every task and config row uses it unchanged.
# Fields not listed here keep the defaults of text.py's EmbedderConfig (pooling="mean_max", sink, use_idf, ...).
# ---------------------------------------------------------------------------------------------------------
LOCKED_CONFIG = dict(
    # 1. Context & architecture
    echo=False,
    layers=(8, 18, 8),            # note: layer 8 is listed twice, so it gets double weight in the layer average
    max_text_tokens=8192,
    max_batch_tokens=1638400,

    # 2. Pooling & post-processing
    mean_weight=0.5,
    half_weight=0.5,
    remove_pcs=0,
    zca_alpha=0.0,
    standardize=True,
    separate_query_std=True,

    # 3. Lexical & fusion
    hybrid_weight=0.1,
    bm25_k1=1.5,
    bm25_b=0.6,
    lex_bigrams=True,

    # 4. Refinements (all off)
    prf_k=0,
    hub_gamma=0.0,
    norm_tau=0.0,

    # 5. Token conditioning
    token_zscore=False,
    mask_outlier_tokens=False,
    token_norm=True,
)


def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _normalize_text(text: Any) -> str:
    """NFKC + whitespace collapse; empty texts become a placeholder (text.py raises on empty)."""
    norm = unicodedata.normalize("NFKC", str(text if text is not None else ""))
    norm = " ".join(norm.strip().split())
    return norm if norm else EMPTY_TEXT_PLACEHOLDER


def _free_device_cache(device: torch.device) -> None:
    gc.collect()
    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()


# =====================================================================
# 1. EMBEDDERS (Qwen via text.py, optional BGE baseline)
# =====================================================================
def make_config(
    model_dir: str,
    dtype: torch.dtype,
    batch_size: int | None = None,
    overrides: dict | None = None,
) -> EmbedderConfig:
    """EmbedderConfig = LOCKED_CONFIG + run settings + optional per-row overrides (empty by default).
    batch_size=None keeps text.py's default."""
    params = {**LOCKED_CONFIG, **(overrides or {})}
    unknown = sorted(set(params) - {f.name for f in dataclasses.fields(EmbedderConfig)})
    if unknown:
        raise ValueError(f"EmbedderConfig in text.py has no field(s) {unknown}; is text.py up to date?")
    if batch_size is not None:
        params["batch_size"] = batch_size
    return EmbedderConfig(model_dir=model_dir, dtype=dtype, **params)


def _param_str(model: torch.nn.Module) -> str:
    total = sum(p.numel() for p in model.parameters())
    return f"{total / 1e9:.2f}B" if total >= 1e9 else f"{total / 1e6:.0f}M"


class BGEEmbedder:
    """Baseline run through the same pipeline (sentence-transformers). Uses BGE's query instruction."""

    def __init__(
        self,
        model_name: str = "BAAI/bge-small-en-v1.5",
        device: str | torch.device | None = None,
        max_length: int = 512,
    ):
        from sentence_transformers import SentenceTransformer

        self.device = torch.device(device) if device else select_device()
        self.model = SentenceTransformer(model_name, device=str(self.device))
        self.model.max_seq_length = max_length
        self.param_str = _param_str(self.model)
        self.model_name = model_name

    def _enc(self, texts: Sequence[str], batch_size: int, show_progress_bar: bool) -> np.ndarray:
        return self.model.encode(
            [_normalize_text(t) for t in texts],
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=show_progress_bar,
        ).astype(np.float32)

    def encode(self, sentences, batch_size: int = 32, show_progress_bar: bool = True, **kwargs):
        return self._enc(sentences, batch_size, show_progress_bar)

    def encode_queries(self, queries, batch_size: int = 32, show_progress_bar: bool = True, **kwargs):
        return self._enc([BGE_QUERY_INSTRUCTION + q for q in queries], batch_size, show_progress_bar)


# =====================================================================
# 2. DATASET LOADER & METRICS (nDCG@10, MRR@5)
# =====================================================================
def _load_split_auto(repo_id: str, config_name: str):
    ds_dict = load_dataset(repo_id, config_name)
    if isinstance(ds_dict, dict):
        for candidate in ("test", "corpus", "queries", "dev", "validation", "train"):
            if candidate in ds_dict:
                return ds_dict[candidate]
        return ds_dict[next(iter(ds_dict.keys()))]
    return ds_dict


def load_mteb_medical_task(task_name: str):
    cfg = BENCHMARK_REGISTRY[task_name]
    repo_id = cfg["hf_repo"]
    logger.info(f"[{task_name}] Loading dataset: '{repo_id}'...")

    corpus_ds = _load_split_auto(repo_id, cfg["corpus_config"])
    queries_ds = _load_split_auto(repo_id, cfg["queries_config"])
    qrels_ds = _load_split_auto(repo_id, cfg["qrels_config"])

    qrels: dict[str, dict[str, int]] = defaultdict(dict)
    for row in qrels_ds:
        qid = str(row.get("query-id", row.get("qid", "")))
        cid = str(row.get("corpus-id", row.get("docid", row.get("pid", ""))))
        score = int(float(row.get("score", 1)))
        if score > 0 and qid and cid:
            qrels[qid][cid] = score

    queries: dict[str, str] = {}
    for row in queries_ds:
        qid = str(row.get("_id", row.get("id", row.get("query-id", ""))))
        if qid in qrels:
            queries[qid] = str(row.get("text", row.get("query", "")))

    corpus_ids: list[str] = []
    corpus_texts: list[str] = []
    for row in corpus_ds:
        cid = str(row.get("_id", row.get("id", row.get("corpus-id", ""))))
        title = str(row.get("title", "") or "").strip()
        text = str(row.get("text", "") or "").strip()
        corpus_ids.append(cid)
        corpus_texts.append(f"{title} {text}".strip() if title else text)

    logger.info(
        f"[{task_name}] Loaded {len(corpus_ids):,} docs | {len(queries):,} active queries | "
        f"{sum(len(v) for v in qrels.values()):,} qrels."
    )
    return corpus_ids, corpus_texts, queries, qrels


def compute_ndcg_at_k(relevances: list[int], ideal_relevances: list[int], k: int = 10) -> float:
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances[:k]) if rel > 0)
    idcg = sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(sorted(ideal_relevances, reverse=True)[:k])
        if rel > 0
    )
    return (dcg / idcg) if idcg > 0.0 else 0.0


def compute_mrr_at_k(relevances: list[int], k: int = 5) -> float:
    for i, rel in enumerate(relevances[:k]):
        if rel > 0:
            return 1.0 / (i + 1)
    return 0.0


@torch.inference_mode()
def _topk_retrieval(
    q_embs: np.ndarray,
    c_embs: np.ndarray,
    device: torch.device,
    top_k: int,
    corpus_chunk_size: int,
) -> np.ndarray:
    """Exact chunked top-k by dot product. Returns [num_queries, top_k] corpus indices (-1 = empty)."""
    num_queries, num_docs = q_embs.shape[0], c_embs.shape[0]
    q_tensor = torch.as_tensor(q_embs, dtype=torch.float32, device=device)
    top_scores = torch.full((num_queries, top_k), -1e9, dtype=torch.float32, device=device)
    top_indices = torch.full((num_queries, top_k), -1, dtype=torch.long, device=device)

    for start_c in range(0, num_docs, corpus_chunk_size):
        end_c = min(start_c + corpus_chunk_size, num_docs)
        c_chunk = torch.as_tensor(c_embs[start_c:end_c], dtype=torch.float32, device=device)
        sim_chunk = q_tensor @ c_chunk.T

        k_chunk = min(top_k, end_c - start_c)
        chunk_scores, chunk_idx = torch.topk(sim_chunk, k=k_chunk, dim=1)
        chunk_idx = chunk_idx + start_c

        combined_scores = torch.cat([top_scores, chunk_scores], dim=1)
        combined_indices = torch.cat([top_indices, chunk_idx], dim=1)
        top_scores, pos = torch.topk(combined_scores, k=min(top_k, combined_scores.size(1)), dim=1)
        top_indices = torch.gather(combined_indices, 1, pos)
        del c_chunk, sim_chunk

    result = top_indices.cpu().numpy()
    del q_tensor, top_scores, top_indices
    return result


def _score_rankings(
    top_indices_np: np.ndarray,
    query_ids: list[str],
    corpus_ids: list[str],
    qrels: dict[str, dict[str, int]],
) -> dict[str, float]:
    """nDCG@10 / MRR@5, ignoring the document whose ID equals the query ID (MTEB behaviour)."""
    ndcg_list, mrr_list = [], []
    for q_idx, qid in enumerate(query_ids):
        gold = qrels[qid]
        ideal_rels = list(gold.values())
        retrieved = [corpus_ids[i] for i in top_indices_np[q_idx] if i >= 0]
        retrieved = [cid for cid in retrieved if cid != qid][:EVAL_TOP_K]
        rels = [gold.get(cid, 0) for cid in retrieved]
        ndcg_list.append(compute_ndcg_at_k(rels, ideal_rels, k=10))
        mrr_list.append(compute_mrr_at_k(rels, k=5))
    return {
        "nDCG@10": round(float(np.mean(ndcg_list) * 100.0), 2),
        "MRR@5": round(float(np.mean(mrr_list) * 100.0), 2),
    }


def evaluate_qwen_task(
    backbone: MedicalTextEmbedder,
    config: EmbedderConfig,
    task_name: str,
    task_data: tuple,
    transductive: bool = True,
) -> tuple[dict[str, Any], str | None]:
    """
    Indexes the task corpus with the full text.py pipeline and scores every query.
    `backbone` only supplies the loaded weights/tokenizer; a fresh MedicalTextEmbedder is built per task so
    IDF, calibration, post-processor and caches never leak from one corpus to the next.
    Returns ({"nDCG@10", "MRR@5", "encode_sec", "transductive"}, optional footnote).
    """
    corpus_ids, corpus_texts, queries_dict, qrels = task_data
    query_ids = list(queries_dict.keys())
    query_texts = [_normalize_text(queries_dict[qid]) for qid in query_ids]
    corpus_texts = [_normalize_text(t) for t in corpus_texts]

    # Query IDs that are also corpus IDs (ArguAna): each query's own document is a trivial hit, which would
    # distort the query-side statistics. Run those tasks corpus-only (queries are not used for fitting).
    overlap = bool(set(query_ids) & set(corpus_ids))
    use_query_set = transductive and not overlap
    note = None
    if transductive and overlap:
        note = f"{task_name}: query IDs overlap corpus IDs -> run corpus-only (queries not used for fitting)"
        logger.warning(f"[{task_name}] {note}")
    cfg = config if use_query_set else replace(config, norm_tau=0.0)   # Sinkhorn needs the query set

    emb = MedicalTextEmbedder(cfg, model=backbone.model, tokenizer=backbone.tokenizer)
    t0 = time.perf_counter()
    logger.info(f"[{task_name}] Indexing {len(corpus_texts):,} documents (+ {len(query_texts):,} queries)...")
    emb.index_corpus(corpus_texts, fit_queries=query_texts if use_query_set else None)
    scores = emb._score_matrix(query_texts)  # [Q, N]: dense + BM25 fusion (per LOCKED_CONFIG)
    encode_sec = time.perf_counter() - t0

    k = min(EVAL_TOP_K + 1, scores.size(1))  # +1 slot so removing an identical-ID hit still leaves 10 results
    top = torch.topk(scores, k=k, dim=1).indices.cpu().numpy()
    out: dict[str, Any] = _score_rankings(top, query_ids, corpus_ids, qrels)
    out["encode_sec"] = round(encode_sec, 1)
    out["transductive"] = use_query_set
    logger.info(
        f"[{task_name}] {encode_sec:.1f}s -> nDCG@10 {out['nDCG@10']:.2f}, MRR@5 {out['MRR@5']:.2f}"
        f" ({'transductive' if use_query_set else 'corpus-only'})"
    )

    del emb, scores, top
    _free_device_cache(backbone.model.device)
    return out, note


def evaluate_dense_baseline(
    embedder: Any,
    task_name: str,
    batch_size: int = 32,
    corpus_chunk_size: int = 25000,
    task_data: tuple | None = None,
) -> dict[str, Any]:
    """Plain cosine retrieval for a standard dense embedder (BGE)."""
    corpus_ids, corpus_texts, queries_dict, qrels = task_data or load_mteb_medical_task(task_name)
    query_ids = list(queries_dict.keys())
    query_texts = [queries_dict[qid] for qid in query_ids]

    t0 = time.perf_counter()
    logger.info(f"[{task_name}] Encoding {len(query_texts):,} queries...")
    q_embs = embedder.encode_queries(query_texts, batch_size=batch_size, show_progress_bar=True)
    logger.info(f"[{task_name}] Encoding {len(corpus_texts):,} documents...")
    c_embs = embedder.encode(corpus_texts, batch_size=batch_size, show_progress_bar=True)
    encode_sec = time.perf_counter() - t0

    top = _topk_retrieval(q_embs, c_embs, embedder.device, EVAL_TOP_K + 1, corpus_chunk_size)
    out: dict[str, Any] = _score_rankings(top, query_ids, corpus_ids, qrels)
    out["encode_sec"] = round(encode_sec, 1)
    logger.info(f"[{task_name}] encode {encode_sec:.1f}s -> nDCG@10 {out['nDCG@10']:.2f}, MRR@5 {out['MRR@5']:.2f}")

    del q_embs, c_embs
    _free_device_cache(embedder.device)
    return out


# =====================================================================
# 3. REPORT FORMATTER
# =====================================================================
def print_and_save_benchmark_table(
    rows: dict[str, dict[str, dict[str, float]]],
    param_strs: dict[str, str],
    output_json: str = "un_clinical_benchmark_results.json",
    output_md: str = "UN_CLINICAL_BENCHMARK_REPORT.md",
    raw_results: dict | None = None,
    footnotes: Sequence[str] = (),
    run_info: dict | None = None,
):
    """
    rows: {row_label: {task: {"nDCG@10", "MRR@5"}}}; param_strs: {row_label: '2.03B'}.
    The published BGE reference row is always printed first.
    """
    name_w = max([len(BASELINE_BGE_SMALL["model_name"])] + [len(k) for k in rows] + [28]) + 1
    header_top = f"{'Model':<{name_w}} | {'# Params':^8} | " + " | ".join(f"{t:^17}" for t in TASKS_ORDER)
    header_sub = f"{'':<{name_w}} | {'':^8} | " + " | ".join(f"{'nDCG@10':>8} {'MRR@5':>8}" for _ in TASKS_ORDER)
    sep = "-" * len(header_top)

    def fmt(name: str, params: str, scores_map: dict) -> str:
        cells = []
        for t in TASKS_ORDER:
            if t in scores_map:
                cells.append(f"{scores_map[t]['nDCG@10']:8.2f} {scores_map[t]['MRR@5']:8.2f}")
            else:
                cells.append(f"{'--':>8} {'--':>8}")
        return f"{name:<{name_w}} | {params:^8} | " + " | ".join(cells)

    print("\n" + "=" * len(header_top))
    print("Medical / Clinical Retrieval Benchmarks".center(len(header_top)))
    print("=" * len(header_top))
    print(header_top)
    print(header_sub)
    print(sep)
    print(fmt(BASELINE_BGE_SMALL["model_name"], BASELINE_BGE_SMALL["params"], BASELINE_BGE_SMALL))
    for label, scores_map in rows.items():
        print(fmt(label, param_strs.get(label, "?"), scores_map))
    print("=" * len(header_top))
    for note in dict.fromkeys(footnotes):
        print(f"* {note}")
    print()

    md = [
        "# Medical / Clinical Retrieval Benchmarks",
        "",
        "| Model | # Params | " + " | ".join(f"{t} (nDCG@10 / MRR@5)" for t in TASKS_ORDER) + " |",
        "| :--- | :---: | " + " | ".join(":---:" for _ in TASKS_ORDER) + " |",
        f"| {BASELINE_BGE_SMALL['model_name']} | {BASELINE_BGE_SMALL['params']} | "
        + " | ".join(
            f"{BASELINE_BGE_SMALL[t]['nDCG@10']:.2f} / {BASELINE_BGE_SMALL[t]['MRR@5']:.2f}" for t in TASKS_ORDER
        )
        + " |",
    ]
    for label, scores_map in rows.items():
        cells = [
            f"{scores_map[t]['nDCG@10']:.2f} / {scores_map[t]['MRR@5']:.2f}" if t in scores_map else "-- / --"
            for t in TASKS_ORDER
        ]
        md.append(f"| **{label}** | {param_strs.get(label, '?')} | " + " | ".join(cells) + " |")
    if footnotes:
        md += [""] + [f"- {note}" for note in dict.fromkeys(footnotes)]

    with open(output_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(
            {
                "published_baseline": BASELINE_BGE_SMALL,
                "rows": rows,
                "params": param_strs,
                "run_info": run_info,
                "footnotes": list(dict.fromkeys(footnotes)),
                "raw_results": raw_results,
            },
            f,
            indent=2,
            default=str,
        )
    logger.info(f"Saved Markdown report to '{output_md}' and JSON audit to '{output_json}'.")


# =====================================================================
# 4. RUNNER
# =====================================================================
DEFAULT_CONFIGS = [
    # (label, transductive, overrides applied on top of LOCKED_CONFIG; leave {} to keep it locked)
    ("Qwen3VL-Text mean_max", True, {}),
    # ("Qwen3VL-Text mean_max (corpus-only)", False, {}),   # queries not used for fitting (no query-side std)
]


def run_evaluation(
    model_dir: str = "./saved_qwen3vl_text_embedder",
    tasks: list[str] | None = None,
    configs: list[tuple[str, bool, dict]] | None = None,
    batch_size: int | None = None,
    dtype: torch.dtype = torch.bfloat16,
    run_bge: bool = False,
    bge_model_name: str = "BAAI/bge-small-en-v1.5",
    bge_batch_size: int = 32,
):
    """
    Loads each dataset once, loads the Qwen weights once, then evaluates every config on every task.
    Row labels are the config labels, e.g. 'Qwen3VL-Text mean_max'.

    TRECCOVID (171k documents) is not in the default task list: the pipeline keeps the raw per-layer states
    (N x 3 x 4096 floats) plus the final embeddings in RAM, roughly 11 GB for that corpus, and encoding it
    at 8192 tokens takes hours on a laptop GPU.
    """
    tasks = tasks or ["MedicalQARetrieval", "PublicHealthQA", "NFCorpus", "ArguAna"]
    configs = configs or DEFAULT_CONFIGS

    task_cache = {t: load_mteb_medical_task(t) for t in tasks}

    first_cfg = make_config(model_dir, dtype, batch_size, configs[0][2])
    backbone = MedicalTextEmbedder(first_cfg)  # loads the weights once; per-task embedders share them
    params = _param_str(backbone.model)
    logger.info(f"Loaded '{model_dir}' ({params}) on {backbone.model.device}")

    rows: dict[str, dict[str, dict[str, float]]] = defaultdict(dict)
    param_strs: dict[str, str] = {}
    raw_results: dict[str, Any] = {}
    run_info: dict[str, Any] = {}
    footnotes: list[str] = []

    for label, transductive, overrides in configs:
        cfg = make_config(model_dir, dtype, batch_size, overrides)
        logger.info(f"=== Config '{label}' (transductive={transductive}, overrides={overrides}) ===")
        run_info[label] = {
            "transductive": transductive,
            "batch_size": cfg.batch_size,
            **{k: getattr(cfg, k) for k in {**LOCKED_CONFIG, **overrides}},
        }
        for task in tasks:
            res, note = evaluate_qwen_task(backbone, cfg, task, task_cache[task], transductive)
            raw_results.setdefault(label, {})[task] = res
            rows[label][task] = {"nDCG@10": res["nDCG@10"], "MRR@5": res["MRR@5"]}
            param_strs[label] = params
            if note:
                footnotes.append(note)

    device = backbone.model.device
    del backbone
    _free_device_cache(device)

    if run_bge:
        bge = BGEEmbedder(model_name=bge_model_name, max_length=512)
        for task in tasks:
            res = evaluate_dense_baseline(bge, task, batch_size=bge_batch_size, task_data=task_cache[task])
            raw_results.setdefault(bge.model_name, {})[task] = res
            row = f"{bge.model_name} (re-run)"
            rows[row][task] = {"nDCG@10": res["nDCG@10"], "MRR@5": res["MRR@5"]}
            param_strs[row] = bge.param_str

    print_and_save_benchmark_table(
        dict(rows), param_strs, raw_results=raw_results, footnotes=footnotes, run_info=run_info
    )
    return dict(rows)



In [ ]:

results = run_evaluation(
    model_dir="madrisight/madrimed1.2-text-embed",
    tasks=["MedicalQARetrieval", "PublicHealthQA", "NFCorpus", "ArguAna"],  # add "TRECCOVID" only with ~12 GB free RAM
)

In [4]:
results

{'Qwen3VL-Text mean_max': {'MedicalQARetrieval': {'nDCG@10': 49.66,
   'MRR@5': 40.75},
  'PublicHealthQA': {'nDCG@10': 77.51, 'MRR@5': 71.04},
  'NFCorpus': {'nDCG@10': 31.79, 'MRR@5': 49.91},
  'ArguAna': {'nDCG@10': 44.42, 'MRR@5': 33.57}}}

In [ ]:

results = run_evaluation(
    model_dir="madrisight/madrimed1.2-text-embed",
    tasks=["TRECCOVID"],
)

In [4]:
results

{'Qwen3VL-Text mean_max': {'TRECCOVID': {'nDCG@10': 52.28, 'MRR@5': 79.9}}}